# NẠP VÀ GHÉP DỮ LIỆU THÔ UIT-VSFC

In [1]:
import os
import sys
import pandas as pd
import numpy as np

In [2]:
#Kiểm tra đường dẫn để chạy được trên cả VS Code và Google Colab
ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
RAW_DIR = os.path.join(ROOT, 'data', 'raw')
PROC_DIR = os.path.join(ROOT, 'data', 'processed')
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROC_DIR, exist_ok=True)

In [3]:
#Nếu chạy trên Colab mà chưa có file raw thì tự tải từ Google Drive của UIT-VSFC
if not os.path.exists(os.path.join(RAW_DIR, 'train_sentences.txt')):
    import gdown
    urls = {
        'train_sentences.txt': '1nzak5OkrheRV1ltOGCXkT671bmjODLhP',
        'train_sentiments.txt': '1ye-gOZIBqXdKOoi_YxvpT6FeRNmViPPv',
        'train_topics.txt': '14MuDtwMnNOcr4z_8KdpxprjbwaQ7lJ_C',
        'dev_sentences.txt': '1sMJSR3oRfPc3fe1gK-V3W5F24tov_517',
        'dev_sentiments.txt': '1GiY1AOp41dLXIIkgES4422AuDwmbUseL',
        'dev_topics.txt': '1DwLgDEaFWQe8mOd7EpF-xqMEbDLfdT-W',
        'test_sentences.txt': '1aNMOeZZbNwSRkjyCWAGtNCMa3YrshR-n',
        'test_sentiments.txt': '1vkQS5gI0is4ACU58-AbWusnemw7KZNfO',
        'test_topics.txt': '1_ArMpDguVsbUGl-xSMkTF_p5KpZrmpSB'
    }
    for fname, fid in urls.items():
        gdown.download(id=fid, output=os.path.join(RAW_DIR, fname), quiet=False)

In [4]:
os.listdir(RAW_DIR)

['dev_sentences.txt',
 'dev_sentiments.txt',
 'dev_topics.txt',
 'test_sentences.txt',
 'test_sentiments.txt',
 'test_topics.txt',
 'train_sentences.txt',
 'train_sentiments.txt',
 'train_topics.txt']

## GHÉP TẬP TRAIN

In [5]:
#Đọc từng dòng file text gốc của tập train
train_sents = open(os.path.join(RAW_DIR, 'train_sentences.txt'), encoding='utf-8').read().splitlines()
train_sentiments = open(os.path.join(RAW_DIR, 'train_sentiments.txt'), encoding='utf-8').read().splitlines()
train_topics = open(os.path.join(RAW_DIR, 'train_topics.txt'), encoding='utf-8').read().splitlines()

In [6]:
print(len(train_sents), len(train_sentiments), len(train_topics))

11426 11426 11426


In [7]:
tempt = train_sents[0]
tempt

'slide giáo trình đầy đủ .'

In [8]:
df_train = pd.DataFrame({
    'sentence': train_sents,
    'sentiment': [int(x) for x in train_sentiments],
    'topic': [int(x) for x in train_topics]
})

In [9]:
df_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 11426 entries, 0 to 11425
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   sentence   11426 non-null  str  
 1   sentiment  11426 non-null  int64
 2   topic      11426 non-null  int64
dtypes: int64(2), str(1)
memory usage: 1.1 MB


In [10]:
df_train.head()

,sentence,sentiment,topic
0,slide giáo trình đầy đủ .,2,1
1,"nhiệt tình giảng dạy , gần gũi với sinh viên .",2,0
2,đi học đầy đủ full điểm chuyên cần .,0,1
3,chưa áp dụng công nghệ thông tin và các thiết ...,0,0
4,"thầy giảng bài hay , có nhiều bài tập ví dụ ng...",2,0


In [11]:
df_train['sentiment'].value_counts()

sentiment
2    5643
0    5325
1     458
Name: count, dtype: int64

In [12]:
df_train['topic'].value_counts()

topic
0    8166
1    2201
3     562
2     497
Name: count, dtype: int64

## GHÉP TẬP DEV VÀ TEST

In [13]:
dev_sents = open(os.path.join(RAW_DIR, 'dev_sentences.txt'), encoding='utf-8').read().splitlines()
dev_sentiments = open(os.path.join(RAW_DIR, 'dev_sentiments.txt'), encoding='utf-8').read().splitlines()
dev_topics = open(os.path.join(RAW_DIR, 'dev_topics.txt'), encoding='utf-8').read().splitlines()

In [14]:
df_dev = pd.DataFrame({
    'sentence': dev_sents,
    'sentiment': [int(x) for x in dev_sentiments],
    'topic': [int(x) for x in dev_topics]
})

In [15]:
df_dev.shape

(1583, 3)

In [16]:
test_sents = open(os.path.join(RAW_DIR, 'test_sentences.txt'), encoding='utf-8').read().splitlines()
test_sentiments = open(os.path.join(RAW_DIR, 'test_sentiments.txt'), encoding='utf-8').read().splitlines()
test_topics = open(os.path.join(RAW_DIR, 'test_topics.txt'), encoding='utf-8').read().splitlines()

In [17]:
df_test = pd.DataFrame({
    'sentence': test_sents,
    'sentiment': [int(x) for x in test_sentiments],
    'topic': [int(x) for x in test_topics]
})

In [18]:
df_test.shape

(3166, 3)

In [19]:
df_test.head()

,sentence,sentiment,topic
0,nói tiếng anh lưu loát .,2,0
1,giáo viên rất vui tính .,2,0
2,cô max có tâm .,2,0
3,"giảng bài thu hút , dí dỏm .",2,0
4,"giáo viên không giảng dạy kiến thức , hướng dẫ...",0,0


## THÊM CỘT NHÃN CHỮ VÀ LƯU FILE CSV

In [20]:
#Ánh xạ mã số sang tên nhãn theo đúng mô tả của UIT-VSFC
sent_map = {0: 'negative', 1: 'neutral', 2: 'positive'}
topic_map = {0: 'lecturer', 1: 'training_program', 2: 'facility', 3: 'others'}

In [21]:
for df in [df_train, df_dev, df_test]:
    df['sentiment_label'] = df['sentiment'].map(sent_map)
    df['topic_label'] = df['topic'].map(topic_map)

In [22]:
df_train.head()

,sentence,sentiment,topic,sentiment_label,topic_label
0,slide giáo trình đầy đủ .,2,1,positive,training_program
1,"nhiệt tình giảng dạy , gần gũi với sinh viên .",2,0,positive,lecturer
2,đi học đầy đủ full điểm chuyên cần .,0,1,negative,training_program
3,chưa áp dụng công nghệ thông tin và các thiết ...,0,0,negative,lecturer
4,"thầy giảng bài hay , có nhiều bài tập ví dụ ng...",2,0,positive,lecturer


In [23]:
df_train.isna().sum()

sentence           0
sentiment          0
topic              0
sentiment_label    0
topic_label        0
dtype: int64

In [24]:
#Lưu ra 3 file csv trong data/processed để các notebook sau đọc lại
df_train.to_csv(os.path.join(PROC_DIR, 'train_raw.csv'), index=False, encoding='utf-8-sig')
df_dev.to_csv(os.path.join(PROC_DIR, 'dev_raw.csv'), index=False, encoding='utf-8-sig')
df_test.to_csv(os.path.join(PROC_DIR, 'test_raw.csv'), index=False, encoding='utf-8-sig')

In [25]:
print('train:', len(df_train), 'dev:', len(df_dev), 'test:', len(df_test))

train: 11426 dev: 1583 test: 3166
